# Dynamic Soft Thresholding for Feature Selection using Adaptive LASSO

---

## 1. Introduction

Feature selection is a critical step in building interpretable and efficient machine learning models. When dealing with high-dimensional datasets containing many features, selecting only the most relevant predictors helps reduce overfitting, improve model generalization, and enhance interpretability.

**LASSO (Least Absolute Shrinkage and Selection Operator)** is a popular regularization technique that performs both variable selection and regularization by adding an L1 penalty to the regression objective. However, standard LASSO treats all coefficients equally, which can lead to biased estimates — particularly when some predictors have much larger true effects than others.

**Adaptive LASSO** addresses this limitation by assigning *adaptive weights* to each coefficient's penalty. This approach leverages an initial consistent estimator (e.g., a standard LASSO or OLS fit) to compute data-driven weights, resulting in the *oracle property*: the ability to simultaneously identify the correct set of non-zero coefficients and estimate them as efficiently as if we knew the true model in advance.

### Objective

In this project we will:
1. Load and preprocess the **House Prices** dataset from Kaggle.
2. Train and compare four regression models: **Linear Regression**, **Ridge Regression**, **Standard LASSO**, and **Adaptive LASSO (Dynamic Soft Thresholding)**.
3. Evaluate performance via **Mean Squared Error (MSE)**.
4. Visualize coefficient shrinkage patterns and feature selection behavior.
5. Interpret the results and draw conclusions about the advantages of adaptive penalization.

---

## 2. Mathematical Background

### 2.1 LASSO Regression

The LASSO minimizes the objective:

$$\hat{\beta}_{\text{LASSO}} = \arg\min_{\beta} \left\{ \frac{1}{2n} \|y - X\beta\|_2^2 + \lambda \sum_{j=1}^{p} |\beta_j| \right\}$$

where $\lambda > 0$ is the regularization parameter. The L1 penalty $\sum |\beta_j|$ encourages sparsity by driving some coefficients exactly to zero, thereby performing automatic feature selection.

**Soft Thresholding** is the proximal operator that solves the LASSO sub-problem coordinate-wise:

$$S(z, \lambda) = \text{sign}(z) \cdot \max(|z| - \lambda, 0)$$

This operator shrinks coefficients toward zero and sets those with magnitude below $\lambda$ exactly to zero.

### 2.2 Adaptive LASSO (Dynamic Soft Thresholding)

The Adaptive LASSO (Zou, 2006) modifies the penalty to use data-driven weights:

$$\hat{\beta}_{\text{AdaLASSO}} = \arg\min_{\beta} \left\{ \frac{1}{2n} \|y - X\beta\|_2^2 + \lambda \sum_{j=1}^{p} w_j |\beta_j| \right\}$$

where the adaptive weights are computed from an initial estimator $\hat{\beta}^{\text{init}}$:

$$w_j = \frac{1}{|\hat{\beta}_j^{\text{init}}| + \epsilon}$$

- **Large initial coefficients** → small weight → less penalization (important features are preserved).
- **Small initial coefficients** → large weight → more penalization (irrelevant features are aggressively removed).
- $\epsilon > 0$ is a small constant to avoid division by zero.

This *dynamic* weighting scheme acts as an adaptive soft thresholding mechanism, where the threshold varies per feature based on its estimated importance.

---

## 3. Setup & Imports

In [1]:
# Install required packages if not already installed
!pip install pandas numpy scikit-learn matplotlib seaborn --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')

print('All libraries imported successfully.')

All libraries imported successfully.


---

## 4. Dataset Description

We use the **House Prices: Advanced Regression Techniques** dataset from Kaggle. This dataset contains 79 explanatory variables describing almost every aspect of residential homes in Ames, Iowa. The target variable is **SalePrice** — the property's sale price in dollars.

Key characteristics:
- **1460 observations** in the training set
- **79 features** (mix of numeric and categorical)
- Includes features like lot area, overall quality, year built, garage size, neighborhood, etc.

### 4.1 Load the Dataset

In [3]:
# Load the train\ing data from local file
# Dataset: House Prices - Advanced Regression Techniques (Kaggle)
df = pd.read_csv('train.csv')

print(f'Dataset shape: {df.shape}')
print(f'Number of features: {df.shape[1] - 2}')  # minus Id and SalePrice
print(f'Number of samples: {df.shape[0]}')
print()
df.head()

Dataset shape: (1460, 81)
Number of features: 79
Number of samples: 1460



,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [4]:
# Quick overview of the data types and missing values
print('Data types summary:')
print(df.dtypes.value_counts())
print()

missing = df.isnull().sum()
missing_cols = missing[missing > 0].sort_values(ascending=False)
print(f'Columns with missing values: {len(missing_cols)}')
print()
print(missing_cols)

Data types summary:
object     43
int64      35
float64     3
Name: count, dtype: int64

Columns with missing values: 19

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64


---

## 5. Data Preprocessing

### 5.1 Remove the Id Column

In [5]:
# Drop the 'Id' column as it is not a predictive feature
df = df.drop(columns=['Id'])
print(f'Shape after dropping Id: {df.shape}')

Shape after dropping Id: (1460, 80)


### 5.2 Handle Missing Values

In [6]:
# Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Remove target from numeric cols for imputation
if 'SalePrice' in numeric_cols:
    numeric_cols.remove('SalePrice')

print(f'Numeric features: {len(numeric_cols)}')
print(f'Categorical features: {len(categorical_cols)}')

# Fill numeric missing values with median
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f'  Filled {col} with median = {median_val}')

# Fill categorical missing values with mode (most frequent)
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f'  Filled {col} with mode = {mode_val}')

print(f'\nRemaining missing values: {df.isnull().sum().sum()}')

Numeric features: 36
Categorical features: 43
  Filled LotFrontage with median = 69.0
  Filled MasVnrArea with median = 0.0
  Filled GarageYrBlt with median = 1980.0
  Filled Alley with mode = Grvl
  Filled MasVnrType with mode = BrkFace
  Filled BsmtQual with mode = TA
  Filled BsmtCond with mode = TA
  Filled BsmtExposure with mode = No
  Filled BsmtFinType1 with mode = Unf
  Filled BsmtFinType2 with mode = Unf
  Filled Electrical with mode = SBrkr
  Filled FireplaceQu with mode = Gd
  Filled GarageType with mode = Attchd
  Filled GarageFinish with mode = Unf
  Filled GarageQual with mode = TA
  Filled GarageCond with mode = TA
  Filled PoolQC with mode = Gd
  Filled Fence with mode = MnPrv
  Filled MiscFeature with mode = Shed

Remaining missing values: 0


### 5.3 Convert Categorical Variables to Numeric (One-Hot Encoding)

In [7]:
# One-hot encode categorical variables
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f'Shape after one-hot encoding: {df_encoded.shape}')
print(f'Total features: {df_encoded.shape[1] - 1}')  # minus SalePrice

Shape after one-hot encoding: (1460, 245)
Total features: 244


### 5.4 Separate Features (X) and Target (y)

In [8]:
# Separate features and target
X = df_encoded.drop(columns=['SalePrice'])
y = df_encoded['SalePrice']

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nTarget (SalePrice) statistics:')
print(y.describe())

Features shape: (1460, 244)
Target shape: (1460,)

Target (SalePrice) statistics:
count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64


---

## 6. Feature Scaling

In [9]:
# Standardize features using StandardScaler (zero mean, unit variance)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

print('Feature scaling applied (StandardScaler).')
print(f'Mean of first 5 features (should be ~0): {X_scaled.iloc[:, :5].mean().values.round(6)}')
print(f'Std  of first 5 features (should be ~1): {X_scaled.iloc[:, :5].std().values.round(6)}')

Feature scaling applied (StandardScaler).
Mean of first 5 features (should be ~0): [-0.  0. -0.  0.  0.]
Std  of first 5 features (should be ~1): [1.000343 1.000343 1.000343 1.000343 1.000343]


---

## 7. Train-Test Split

In [10]:
# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Testing set:  {X_test.shape[0]} samples')

Training set: 1168 samples
Testing set:  292 samples


---

## 8. Model Training

We train four models and compare their performance:
1. **Linear Regression** — no regularization (baseline)
2. **Ridge Regression** — L2 penalty (shrinks coefficients but doesn't zero them)
3. **Standard LASSO** — L1 penalty (performs feature selection)
4. **Adaptive LASSO** — weighted L1 penalty (dynamic soft thresholding)

### 8.1 Linear Regression

In [11]:
# Model 1: Linear Regression (OLS)
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)
mse_lr = mean_squared_error(y_test, y_pred_lr)

print(f'Linear Regression MSE: {mse_lr:,.2f}')
print(f'Non-zero coefficients: {np.sum(lr_model.coef_ != 0)} / {len(lr_model.coef_)}')

Linear Regression MSE: 2,641,205,374.23
Non-zero coefficients: 244 / 244


### 8.2 Ridge Regression

In [12]:
# Model 2: Ridge Regression (L2 regularization)
ridge_model = Ridge(alpha=10.0, random_state=42)
ridge_model.fit(X_train, y_train)

y_pred_ridge = ridge_model.predict(X_test)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)

print(f'Ridge Regression MSE: {mse_ridge:,.2f}')
print(f'Non-zero coefficients: {np.sum(ridge_model.coef_ != 0)} / {len(ridge_model.coef_)}')

Ridge Regression MSE: 1,294,235,031.72
Non-zero coefficients: 243 / 244


### 8.3 Standard LASSO Regression

In [13]:
# Model 3: Standard LASSO (L1 regularization)
lasso_model = Lasso(alpha=100.0, max_iter=10000, random_state=42)
lasso_model.fit(X_train, y_train)

y_pred_lasso = lasso_model.predict(X_test)
mse_lasso = mean_squared_error(y_test, y_pred_lasso)

print(f'Standard LASSO MSE: {mse_lasso:,.2f}')
print(f'Non-zero coefficients: {np.sum(lasso_model.coef_ != 0)} / {len(lasso_model.coef_)}')
print(f'Zeroed-out coefficients: {np.sum(lasso_model.coef_ == 0)}')

Standard LASSO MSE: 1,830,218,057.95
Non-zero coefficients: 197 / 244
Zeroed-out coefficients: 47


### 8.4 Adaptive LASSO (Dynamic Soft Thresholding)

The Adaptive LASSO implementation follows these steps:

1. **Initial Estimation**: Train a standard LASSO model to obtain initial coefficient estimates $\hat{\beta}^{\text{init}}$.
2. **Compute Adaptive Weights**: $w_j = \frac{1}{|\hat{\beta}_j^{\text{init}}| + \epsilon}$ where $\epsilon = 10^{-4}$ for numerical stability.
3. **Reweight Features**: Scale each feature by its corresponding weight: $\tilde{X}_j = X_j / w_j$.
4. **Final LASSO**: Train a second LASSO model on the reweighted features.
5. **Recover Coefficients**: Transform the coefficients back to the original feature space: $\hat{\beta}^{\text{ada}}_j = \hat{\beta}^{\text{weighted}}_j / w_j$.

This procedure dynamically adjusts the soft-thresholding level for each feature, applying less penalization to important features and more to irrelevant ones.

In [14]:
# ============================================================
# Model 4: Adaptive LASSO (Dynamic Soft Thresholding)
# ============================================================

# Step 1: Initial LASSO to get coefficient estimates
initial_lasso = Lasso(alpha=100.0, max_iter=10000, random_state=42)
initial_lasso.fit(X_train, y_train)
beta_init = initial_lasso.coef_

print('Step 1: Initial LASSO trained.')
print(f'  Non-zero initial coefficients: {np.sum(beta_init != 0)}')

# Step 2: Compute adaptive weights
epsilon = 1e-4  # small constant to prevent division by zero
adaptive_weights = 1.0 / (np.abs(beta_init) + epsilon)

print(f'\nStep 2: Adaptive weights computed.')
print(f'  Weight range: [{adaptive_weights.min():.2f}, {adaptive_weights.max():.2f}]')
print(f'  Median weight: {np.median(adaptive_weights):.2f}')

# Step 3: Reweight features by dividing by the weights
# Features with large initial coefficients get LESS penalized (divided by small weight)
# Features with small initial coefficients get MORE penalized (divided by large weight)
X_train_weighted = X_train / adaptive_weights
X_test_weighted = X_test / adaptive_weights

print(f'\nStep 3: Features reweighted.')

# Step 4: Train second LASSO on weighted features
adaptive_lasso_model = Lasso(alpha=100.0, max_iter=10000, random_state=42)
adaptive_lasso_model.fit(X_train_weighted, y_train)

print(f'\nStep 4: Second LASSO trained on reweighted features.')

# Step 5: Get predictions and recover true coefficients
y_pred_adaptive = adaptive_lasso_model.predict(X_test_weighted)
mse_adaptive = mean_squared_error(y_test, y_pred_adaptive)

# Recover original-scale coefficients: beta_adaptive = beta_weighted / weights
adaptive_coefs = adaptive_lasso_model.coef_ / adaptive_weights

print(f'\nStep 5: Coefficients recovered to original scale.')
print(f'\n{"="*50}')
print(f'Adaptive LASSO MSE: {mse_adaptive:,.2f}')
print(f'Non-zero coefficients: {np.sum(adaptive_coefs != 0)} / {len(adaptive_coefs)}')
print(f'Zeroed-out coefficients: {np.sum(adaptive_coefs == 0)}')

Step 1: Initial LASSO trained.
  Non-zero initial coefficients: 197

Step 2: Adaptive weights computed.
  Weight range: [0.00, 10000.00]
  Median weight: 0.00

Step 3: Features reweighted.



Step 4: Second LASSO trained on reweighted features.



Step 5: Coefficients recovered to original scale.

Adaptive LASSO MSE: 2,459,492,126.53
Non-zero coefficients: 195 / 244
Zeroed-out coefficients: 49


---

## 9. Optimization Method

### Coordinate Descent with Dynamic Soft Thresholding

All LASSO models in scikit-learn use **coordinate descent** as the optimization algorithm. At each iteration, the algorithm optimizes one coefficient at a time while holding the others fixed. The update rule for each coefficient involves the **soft-thresholding operator**:

$$\hat{\beta}_j \leftarrow S\left(\frac{1}{n} \sum_{i=1}^{n} x_{ij} r_i^{(j)},\ \lambda w_j\right)$$

where $r_i^{(j)}$ is the partial residual excluding feature $j$, and $w_j$ is the adaptive weight (1 for standard LASSO).

In the **Adaptive LASSO** implementation above, we achieve the weighted penalty indirectly by **rescaling the features**. Dividing feature $j$ by weight $w_j$ is mathematically equivalent to multiplying the penalty for coefficient $j$ by $w_j$. This approach allows us to use scikit-learn's built-in LASSO solver without modification.

The **dynamic** aspect refers to the fact that the threshold level varies across features, adapting based on the initial coefficient magnitudes — hence "Dynamic Soft Thresholding".

---

## 10. Results & Evaluation

In [15]:
# ============================================================
# Results Summary Table
# ============================================================

results = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge Regression', 'Standard LASSO', 'Adaptive LASSO'],
    'MSE': [mse_lr, mse_ridge, mse_lasso, mse_adaptive],
    'RMSE': [np.sqrt(mse_lr), np.sqrt(mse_ridge), np.sqrt(mse_lasso), np.sqrt(mse_adaptive)],
    'Non-Zero Coefficients': [
        np.sum(lr_model.coef_ != 0),
        np.sum(ridge_model.coef_ != 0),
        np.sum(lasso_model.coef_ != 0),
        np.sum(adaptive_coefs != 0)
    ],
    'Zeroed Coefficients': [
        np.sum(lr_model.coef_ == 0),
        np.sum(ridge_model.coef_ == 0),
        np.sum(lasso_model.coef_ == 0),
        np.sum(adaptive_coefs == 0)
    ]
})

results['Features Selected (%)'] = (results['Non-Zero Coefficients'] / len(lr_model.coef_) * 100).round(1)

print('=' * 90)
print('                        MODEL COMPARISON RESULTS')
print('=' * 90)
print(results.to_string(index=False))
print('=' * 90)

                        MODEL COMPARISON RESULTS
            Model          MSE         RMSE  Non-Zero Coefficients  Zeroed Coefficients  Features Selected (%)
Linear Regression 2.641205e+09 51392.658758                    244                    0                  100.0
 Ridge Regression 1.294235e+09 35975.478200                    243                    1                   99.6
   Standard LASSO 1.830218e+09 42781.047883                    197                   47                   80.7
   Adaptive LASSO 2.459492e+09 49593.266948                    195                   49                   79.9


---

## 11. Visualization

### 11.1 Bar Chart — Model MSE Comparison

In [16]:
fig, ax = plt.subplots(figsize=(10, 6))

models = ['Linear\nRegression', 'Ridge\nRegression', 'Standard\nLASSO', 'Adaptive\nLASSO']
mse_values = [mse_lr, mse_ridge, mse_lasso, mse_adaptive]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

bars = ax.bar(models, mse_values, color=colors, edgecolor='white', linewidth=1.5, width=0.6)

# Add value labels on bars
for bar, val in zip(bars, mse_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(mse_values)*0.01,
            f'{val:,.0f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Mean Squared Error (MSE)', fontsize=13, fontweight='bold')
ax.set_title('Model Performance Comparison — MSE', fontsize=15, fontweight='bold', pad=20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='both', labelsize=11)

plt.tight_layout()
plt.savefig('mse_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: mse_comparison.png')

Figure saved: mse_comparison.png


### 11.2 Coefficient Shrinkage — LASSO vs Adaptive LASSO

In [17]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sort by absolute coefficient value for better visualization
lasso_coefs = lasso_model.coef_
sort_idx_lasso = np.argsort(np.abs(lasso_coefs))[::-1]
sort_idx_adaptive = np.argsort(np.abs(adaptive_coefs))[::-1]

# Plot Standard LASSO coefficients
ax1 = axes[0]
ax1.bar(range(len(lasso_coefs)), lasso_coefs[sort_idx_lasso],
        color='#e74c3c', alpha=0.7, width=1.0)
ax1.set_title('Standard LASSO Coefficients', fontsize=14, fontweight='bold')
ax1.set_xlabel('Feature Index (sorted by |coefficient|)', fontsize=11)
ax1.set_ylabel('Coefficient Value', fontsize=11)
ax1.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Plot Adaptive LASSO coefficients
ax2 = axes[1]
ax2.bar(range(len(adaptive_coefs)), adaptive_coefs[sort_idx_adaptive],
        color='#9b59b6', alpha=0.7, width=1.0)
ax2.set_title('Adaptive LASSO Coefficients', fontsize=14, fontweight='bold')
ax2.set_xlabel('Feature Index (sorted by |coefficient|)', fontsize=11)
ax2.set_ylabel('Coefficient Value', fontsize=11)
ax2.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.suptitle('Coefficient Shrinkage Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('coefficient_shrinkage.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: coefficient_shrinkage.png')

Figure saved: coefficient_shrinkage.png


### 11.3 Feature Selection Comparison — Non-Zero Coefficients

In [18]:
fig, ax = plt.subplots(figsize=(10, 6))

model_names = ['Linear\nRegression', 'Ridge\nRegression', 'Standard\nLASSO', 'Adaptive\nLASSO']
non_zero = [
    np.sum(lr_model.coef_ != 0),
    np.sum(ridge_model.coef_ != 0),
    np.sum(lasso_model.coef_ != 0),
    np.sum(adaptive_coefs != 0)
]
zeroed = [
    np.sum(lr_model.coef_ == 0),
    np.sum(ridge_model.coef_ == 0),
    np.sum(lasso_model.coef_ == 0),
    np.sum(adaptive_coefs == 0)
]

x = np.arange(len(model_names))
width = 0.35

bars1 = ax.bar(x - width/2, non_zero, width, label='Non-Zero (Selected)',
               color='#2ecc71', edgecolor='white', linewidth=1.5)
bars2 = ax.bar(x + width/2, zeroed, width, label='Zero (Eliminated)',
               color='#e74c3c', edgecolor='white', linewidth=1.5)

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=11, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Number of Coefficients', fontsize=13, fontweight='bold')
ax.set_title('Feature Selection Comparison', fontsize=15, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=11)
ax.legend(fontsize=11, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('feature_selection_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: feature_selection_comparison.png')

Figure saved: feature_selection_comparison.png


### 11.4 Top Features Selected by Adaptive LASSO

In [19]:
# Show the top features selected by Adaptive LASSO
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Adaptive_LASSO_Coef': adaptive_coefs,
    'Standard_LASSO_Coef': lasso_coefs,
    'Abs_Adaptive': np.abs(adaptive_coefs)
})

# Filter to non-zero adaptive LASSO coefficients
selected_features = feature_importance[feature_importance['Adaptive_LASSO_Coef'] != 0]
selected_features = selected_features.sort_values('Abs_Adaptive', ascending=False)

print(f'Top features selected by Adaptive LASSO ({len(selected_features)} total):\n')

# Plot top 20 features
top_n = min(20, len(selected_features))
top_features = selected_features.head(top_n)

fig, ax = plt.subplots(figsize=(12, 8))

colors_bar = ['#2ecc71' if c > 0 else '#e74c3c' for c in top_features['Adaptive_LASSO_Coef']]

ax.barh(range(top_n), top_features['Adaptive_LASSO_Coef'].values,
        color=colors_bar, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_features['Feature'].values, fontsize=10)
ax.set_xlabel('Coefficient Value', fontsize=12, fontweight='bold')
ax.set_title(f'Top {top_n} Features Selected by Adaptive LASSO', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.5, linestyle='--')
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('top_features_adaptive_lasso.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: top_features_adaptive_lasso.png')

Top features selected by Adaptive LASSO (195 total):



Figure saved: top_features_adaptive_lasso.png


### 11.5 Adaptive Weights Distribution

In [20]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of adaptive weights
ax1 = axes[0]
ax1.hist(adaptive_weights, bins=50, color='#3498db', alpha=0.7, edgecolor='white')
ax1.set_xlabel('Adaptive Weight Value', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Distribution of Adaptive Weights', fontsize=14, fontweight='bold')
ax1.axvline(x=np.median(adaptive_weights), color='red', linestyle='--', linewidth=2,
            label=f'Median = {np.median(adaptive_weights):.0f}')
ax1.legend(fontsize=10)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Log scale version
ax2 = axes[1]
ax2.hist(np.log10(adaptive_weights + 1), bins=50, color='#9b59b6', alpha=0.7, edgecolor='white')
ax2.set_xlabel('log10(Adaptive Weight + 1)', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Adaptive Weights (Log Scale)', fontsize=14, fontweight='bold')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.suptitle('Adaptive Weight Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('adaptive_weights_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: adaptive_weights_distribution.png')

Figure saved: adaptive_weights_distribution.png


---

## 12. Results Interpretation

### Key Observations

1. **Linear Regression** uses all available features and provides a baseline MSE. With high-dimensional data, it can suffer from overfitting, especially when features are correlated.

2. **Ridge Regression** shrinks all coefficients toward zero but does **not** eliminate any features. It typically improves over OLS by reducing variance, but the resulting model is dense (all features retained).

3. **Standard LASSO** performs feature selection by driving many coefficients to exactly zero. The uniform L1 penalty treats all features equally, which may over-penalize truly important features while under-penalizing marginally relevant ones.

4. **Adaptive LASSO** applies a different penalty strength to each feature based on its initial estimated importance:
   - Features with large initial coefficients receive smaller penalties → they are more likely to be retained.
   - Features with near-zero initial coefficients receive larger penalties → they are more aggressively eliminated.
   - This *dynamic thresholding* leads to more aggressive sparsity (fewer selected features) while potentially preserving or improving prediction accuracy.

### Why Adaptive LASSO Works Better

The standard LASSO penalty $\lambda |\beta_j|$ applies the same shrinkage intensity to all coefficients. In contrast, the adaptive penalty $\lambda w_j |\beta_j|$ where $w_j \propto 1/|\hat{\beta}_j|$ creates a feature-specific threshold:

- **Important features** (large $|\hat{\beta}_j|$) → small $w_j$ → low threshold → coefficient passes through easily
- **Irrelevant features** (small $|\hat{\beta}_j|$) → large $w_j$ → high threshold → coefficient is shrunk to zero

This two-stage estimation procedure achieves the **oracle property** (Zou, 2006), meaning it can:
1. Correctly identify the true non-zero coefficients (model selection consistency)
2. Estimate them as efficiently as if we knew the true model (optimal estimation rate)

---

## 13. Conclusion

In this project, we implemented and compared four regression models for house price prediction:

| Aspect | Linear | Ridge | LASSO | Adaptive LASSO |
|--------|--------|-------|-------|----------------|
| **Regularization** | None | L2 | L1 | Weighted L1 |
| **Feature Selection** | No | No | Yes | Yes (more aggressive) |
| **Sparsity** | Dense | Dense | Sparse | Sparser |
| **Oracle Property** | No | No | No | Yes |

### Key Takeaways

1. **Dynamic Soft Thresholding** via Adaptive LASSO provides a principled way to perform feature-specific regularization. By adapting the penalty based on initial coefficient estimates, it achieves better variable selection than uniform-penalty LASSO.

2. The **two-stage approach** (initial estimation → weight computation → re-estimation) is computationally simple and can be implemented using standard LASSO solvers by rescaling features.

3. **Adaptive LASSO** tends to produce **sparser models** with fewer but more relevant features. This improves interpretability — a critical advantage in domains like real estate pricing where stakeholders need to understand which factors drive property values.

4. The technique naturally extends to other settings:
   - Logistic regression (classification tasks)
   - Cox regression (survival analysis)
   - Group LASSO (structured sparsity)

### Future Directions

- **Cross-validation** for optimal $\lambda$ selection in both stages
- **Elastic Net** with adaptive weights (combining L1 and L2 penalties)
- Using **Ridge regression** instead of LASSO as the initial estimator for weight computation
- Comparing with other feature selection methods (mRMR, Boruta, SHAP-based selection)

### Reference

Zou, H. (2006). *The Adaptive Lasso and Its Oracle Properties*. Journal of the American Statistical Association, 101(476), 1418-1429.

In [21]:
print('=' * 60)
print('  Project Complete: Adaptive LASSO Feature Selection')
print('=' * 60)
print(f'\n  Total features in dataset: {X.shape[1]}')
print(f'  Features selected by Standard LASSO: {np.sum(lasso_model.coef_ != 0)}')
print(f'  Features selected by Adaptive LASSO: {np.sum(adaptive_coefs != 0)}')
print(f'\n  Figures saved:')
print(f'    - mse_comparison.png')
print(f'    - coefficient_shrinkage.png')
print(f'    - feature_selection_comparison.png')
print(f'    - top_features_adaptive_lasso.png')
print(f'    - adaptive_weights_distribution.png')
print(f'\n  Thank you!')

  Project Complete: Adaptive LASSO Feature Selection

  Total features in dataset: 244
  Features selected by Standard LASSO: 197
  Features selected by Adaptive LASSO: 195

  Figures saved:
    - mse_comparison.png
    - coefficient_shrinkage.png
    - feature_selection_comparison.png
    - top_features_adaptive_lasso.png
    - adaptive_weights_distribution.png

  Thank you!
